Agent
- LLM
- tools
- agent

Built-in LangChain tools: llm-math and wikipedia

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [3]:
### 使用langchain内置工具

from langchain_classic.agents import load_tools, initialize_agent
from langchain_classic.agents import AgentType
from langchain_ollama import ChatOllama

In [4]:
# 新建一个基本的LLM
llm = ChatOllama(model=os.getenv("MODEL_NAME"), temperature=0)

In [8]:
# 初始化工具
tools = load_tools(
    ["llm-math", "wikipedia"],
    llm=llm,
)

In [9]:
# 初始化agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True, #处理解析错误
    verbose=True,
)

C:\Users\61075\AppData\Local\Temp\ipykernel_1644\2018644216.py:2: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [11]:
try:
    agent.invoke(input="who won the 2022 world cup")
except:
    print("exception on external access")



> Entering new AgentExecutor chain...
Question: who won the 2022 world cup
Thought: I need to search Wikipedia for information about the 2022 World Cup winner.
Action:
```
{
  "action": "wikipedia",
  "action_input": "2022 FIFA World Cup"
}
```

Observation: Page: 2022 FIFA World Cup
Summary: The 2022 FIFA World Cup was the 22nd FIFA World Cup, the quadrennial world championship for national football teams organised by FIFA. It took place in Qatar from 20 November to 18 December 2022, after the country was awarded the hosting rights in 2010. It was the first World Cup to be held in the Middle East and the Arabian Peninsula, and the second in an Asian country after the 2002 tournament in South Korea and Japan.
This tournament was the last with 32 participating teams, with the number of teams being increased to 48 for the 2026 World Cup. To avoid the extremes of Qatar's hot and humid climate in summers, the event was held in November and December, becoming the first one to take place i

In [12]:
try:
    agent.invoke(input="calculate 25 percent of 300")
except:
    print("exception on external access")



> Entering new AgentExecutor chain...
Thought: This is a simple percentage calculation, we can use the calculator tool for this.
Action:
```
{
  "action": "Calculator",
  "action_input": "300 * 0.25"
}
```
Observation: Answer: 75.0
Thought:The result from the calculator shows that 25 percent of 300 is 75.

Final Answer: 75

> Finished chain.


In [13]:
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"

try:
    result = agent.invoke(question)
except:
    print("exception on external access")



> Entering new AgentExecutor chain...
Thought: I need to search Wikipedia for information about Tom M. Mitchell's books.
Action:
```
{
  "action": "wikipedia",
  "action_input": "Tom M. Mitchell books"
}
```
Observation: Page: Tom M. Mitchell
Summary: Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning. He is a member of the United States National Academy of Engineering since 2010. He is also a Fellow of the American Academy of Arts and Sciences, the American Association for the Advancement of Science and a Fellow and past president of the Association for the Advancement of Artificial Intelligence. In October 2018, Mitchell was app

In [14]:
result

{'input': 'Tom M. Mitchell is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU)what book did he write?',
 'output': 'The book written by Tom M. Mitchell is "Machine Learning."'}

PythonREPLTool：使agent可以执行python交互式编写代码执行代码

In [18]:
### PythonREPLTool：read-eval-print loop
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_experimental.tools.python.tool import PythonREPLTool

agent = create_python_agent(
    llm=llm,
    tool=PythonREPLTool(),  #使用Python代码交互工具（REPLTool）
    verbose=True
)

In [20]:
# 问题：对名字list排序
customer_list = [["Harrison", "Chase"],
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"],
                 ["Geoff","Fusion"],
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]


agent.invoke(f"""Sort these customers by \
last name and then first name \
and print the output: {customer_list}""")



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


I need to sort the list of customers by their last name and then by first name. I can use Python's `sorted` function with a custom key that extracts the last and first names.
Action: Python_REPL
Action Input: customers = [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]
sorted_customers = sorted(customers, key=lambda x: (x[1], x[0]))
print(sorted_customers)
Observation: [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]

Thought:I now know the final answer
Final Answer: The customers sorted by last name and then first name are as follows:

1. Jen Ayai
2. Lang Chain
3. Harrison Chase
4. Elle Elem
5. Trance Former
6. Geoff Fusion
7. Dolly Too

> Finished chain.


{'input': "Sort these customers by last name and then first name and print the output: [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']]",
 'output': 'The customers sorted by last name and then first name are as follows:\n\n1. Jen Ayai\n2. Lang Chain\n3. Harrison Chase\n4. Elle Elem\n5. Trance Former\n6. Geoff Fusion\n7. Dolly Too'}

自定义工具

In [15]:
### @tool 装饰器可以应用于任何函数，将函数转化为langchain工具，使其成为agent可调用的tool

# 创建自定义时间工具，返回今天日期
from langchain_classic.agents import tool
from datetime import date

@tool
def time(text: str) -> str:
    """Returns today's date, use this for any \
    questions related to knowing today's date. \
    The input should always be an empty string, \
    and this function will always return today's \
    date - any date mathmatics should occur \
    outside this function."""
    return str(date.today())

In [16]:
# 初始化代理
agent= initialize_agent(
    tools=[time], #将刚刚创建的时间工具加入代理
    llm=llm, #初始化的模型
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,  #代理类型
    handle_parsing_errors=True, #处理解析错误
    verbose = True #输出中间步骤
)

In [17]:
try:
    result = agent.invoke(input="whats the date today?")
except:
    print("exception on external access")



> Entering new AgentExecutor chain...
Question: whats the date today?
Thought: I need to get today's date.
Action:
```
{
  "action": "time",
  "action_input": ""
}
```
Observation: 2026-03-09
Thought:I now know the final answer
Final Answer: The date today is 2026-03-09.

> Finished chain.
